In [2]:
import os
import uuid
import random
from datetime import datetime, timedelta
import numpy as np
import pandas as pd

# Enforce deterministic generation
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

OUTPUT_DIR = "pharma_data_warehouse"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------------------------------------------------
# Configuration Parameters
# ---------------------------------------------------------
NUM_HCPS = 250
NUM_PAYERS = 8
NUM_PATIENTS = 2500
NUM_ENGAGEMENTS = 6000
DAYS_HISTORY = 180

START_DATE = datetime.now() - timedelta(days=DAYS_HISTORY)

SPECIALTIES = ["Medical Oncology", "Hematology-Oncology", "Thoracic Oncology"]
TERRITORIES = ["US-EAST-01", "US-EAST-02", "US-CENTRAL-01", "US-WEST-01"]
CHANNELS = ["Rep_F2F", "Rep_Virtual", "Approved_Email", "Webinar", "Digital_Ad"]
TOPICS = ["Efficacy_OS_PFS", "Safety_Adverse_Events", "Dosing_Administration", "Copay_Access_Support"]

print("1/5 Generating HCP Dimension...")
# ---------------------------------------------------------
# 1. HCP Dimension (dim_hcp)
# ---------------------------------------------------------
hcps = []
for i in range(1, NUM_HCPS + 1):
    npi = 1000000000 + i
    hcp_id = f"HCP_{npi}"
    # Power-law decile distribution: 80% volume concentrated in deciles 7-10
    decile = int(np.random.choice(range(1, 11), p=[0.05, 0.05, 0.05, 0.05, 0.08, 0.12, 0.15, 0.18, 0.17, 0.10]))
    channel_pref = np.random.choice(CHANNELS, p=[0.35, 0.20, 0.25, 0.15, 0.05])
    
    hcps.append({
        "hcp_id": hcp_id,
        "npi": npi,
        "first_name": f"Physician_{i}",
        "specialty": np.random.choice(SPECIALTIES, p=[0.55, 0.35, 0.10]),
        "market_decile": decile,
        "territory_id": np.random.choice(TERRITORIES),
        "preferred_channel": channel_pref,
        "email_opt_in": bool(np.random.choice([True, False], p=[0.88, 0.12])),
        "clinical_trial_investigator": bool(np.random.choice([True, False], p=[0.15, 0.85]))
    })
df_hcp = pd.DataFrame(hcps)
df_hcp.to_csv(f"{OUTPUT_DIR}/dim_hcp.csv", index=False)


print("2/5 Generating Payer & Formulary Dimension...")
# ---------------------------------------------------------
# 2. Payer & Formulary Dimension (dim_payer)
# ---------------------------------------------------------
payer_names = [
    ("UnitedHealth_Commercial", "Commercial", 2, True),
    ("Anthem_BCBS", "Commercial", 2, True),
    ("CVS_Caremark_Commercial", "Commercial", 1, False), # Preferred, No PA
    ("Aetna_Commercial", "Commercial", 3, True),       # Restricted
    ("Cigna_Commercial", "Commercial", 2, True),
    ("CMS_Medicare_Part_D", "Medicare", 2, True),
    ("Humana_Medicare", "Medicare", 2, True),
    ("State_Medicaid_Pool", "Medicaid", 3, True)
]

payers = []
for p_idx, (name, plan_type, tier, pa_required) in enumerate(payer_names, start=101):
    payer_id = f"PAYER_{p_idx}"
    payers.append({
        "payer_id": payer_id,
        "payer_name": name,
        "plan_type": plan_type,
        "formulary_tier": tier,            # Tier 1 = Lowest Copay, Tier 3 = High Copay
        "pa_required": pa_required,        # Prior Authorization requirement
        "step_therapy_required": True if tier == 3 else False,
        "brand_coverage_status": "Preferred" if tier == 1 else ("Covered" if tier == 2 else "Restricted")
    })
df_payer = pd.DataFrame(payers)
df_payer.to_csv(f"{OUTPUT_DIR}/dim_payer.csv", index=False)


print("3/5 Generating Patient Master Cohort...")
# ---------------------------------------------------------
# 3. Patient Master Cohort (dim_patient)
# ---------------------------------------------------------
patients = []
hcp_ids = df_hcp["hcp_id"].values
payer_ids = df_payer["payer_id"].values

# Skew patient distribution towards high-decile HCPs
hcp_weights = df_hcp["market_decile"].values / df_hcp["market_decile"].sum()

for i in range(1, NUM_PATIENTS + 1):
    pat_id = f"PAT_{uuid.uuid4().hex[:10].upper()}"
    assigned_hcp = np.random.choice(hcp_ids, p=hcp_weights)
    assigned_payer = np.random.choice(payer_ids)
    age = int(np.clip(np.random.normal(63, 10), 30, 89))
    gender = np.random.choice(["M", "F"], p=[0.52, 0.48])
    
    patients.append({
        "patient_token": pat_id,
        "assigned_hcp_id": assigned_hcp,
        "payer_id": assigned_payer,
        "age": age,
        "gender": gender,
        "ecog_performance_status": int(np.random.choice([0, 1, 2, 3], p=[0.4, 0.35, 0.2, 0.05])),
        "disease_stage": np.random.choice(["Stage_IIIb", "Stage_IV_Metastatic"], p=[0.3, 0.7]),
        "diagnosis_date": (START_DATE - timedelta(days=random.randint(30, 400))).date().isoformat()
    })
df_patient = pd.DataFrame(patients)
df_patient.to_csv(f"{OUTPUT_DIR}/dim_patient.csv", index=False)


print("4/5 Simulating Longitudinal Prescribing & Claims...")
# ---------------------------------------------------------
# 4. Longitudinal Claims & Pharmacy Events (fct_claims)
# ---------------------------------------------------------
claims = []
claim_drugs = ["Our_Brand", "Competitor_A", "Competitor_B"]

# Fast lookup dicts for payer and patient properties
payer_dict = df_payer.set_index("payer_id").to_dict(orient="index")

for pat in patients:
    p_id = pat["patient_token"]
    h_id = pat["assigned_hcp_id"]
    payer_meta = payer_dict[pat["payer_id"]]
    
    # Simulate recurring script visits every 21 or 28 days (chemo/immunology cycles)
    curr_time = START_DATE + timedelta(days=random.randint(0, 30))
    cycle_days = random.choice([21, 28])
    
    while curr_time <= datetime.now():
        claim_id = f"CLM_{uuid.uuid4().hex[:8].upper()}"
        drug = np.random.choice(claim_drugs, p=[0.40, 0.35, 0.25])
        
        # Payer friction logic
        status = "PAID"
        rejection_reason = "NONE"
        copay_amount = float(payer_meta["formulary_tier"] * 25.0)
        
        if drug == "Our_Brand":
            if payer_meta["formulary_tier"] == 3 and np.random.rand() < 0.35:
                status = "REJECTED"
                rejection_reason = "PRIOR_AUTH_DENIAL"
                copay_amount = 0.0
            elif payer_meta["step_therapy_required"] and np.random.rand() < 0.20:
                status = "REJECTED"
                rejection_reason = "STEP_THERAPY_FAIL"
                copay_amount = 0.0
        
        claims.append({
            "claim_id": claim_id,
            "patient_token": p_id,
            "prescribing_hcp_id": h_id,
            "payer_id": pat["payer_id"],
            "claim_date": curr_time.date().isoformat(),
            "ndc_product": drug,
            "claim_status": status,
            "rejection_reason": rejection_reason,
            "patient_copay_amount": copay_amount,
            "days_supply": cycle_days
        })
        curr_time += timedelta(days=cycle_days)

df_claims = pd.DataFrame(claims)
df_claims.to_csv(f"{OUTPUT_DIR}/fct_claims.csv", index=False)


print("5/5 Simulating Omnichannel Field & Digital Engagement...")
# ---------------------------------------------------------
# 5. Omnichannel Engagement Stream (fct_engagement)
# ---------------------------------------------------------
engagements = []
for _ in range(NUM_ENGAGEMENTS):
    hcp_sample = df_hcp.sample(1).iloc[0]
    h_id = hcp_sample["hcp_id"]
    channel = np.random.choice(CHANNELS, p=[0.35, 0.15, 0.30, 0.15, 0.05])
    topic = np.random.choice(TOPICS)
    
    # Event timestamp within the history window
    event_timestamp = START_DATE + timedelta(
        days=random.randint(0, DAYS_HISTORY), 
        hours=random.randint(8, 17), 
        minutes=random.randint(0, 59)
    )
    
    # Channel-specific behavior
    duration_secs = 0
    feedback_sentiment = None
    samples_dropped = 0
    
    if channel in ["Rep_F2F", "Rep_Virtual"]:
        duration_secs = int(np.random.normal(300, 90)) # ~5 minutes
        feedback_sentiment = np.random.choice(["Positive", "Neutral", "Challenging_Efficacy", "Access_Objection"], p=[0.5, 0.25, 0.15, 0.10])
        samples_dropped = int(np.random.choice([0, 2, 4], p=[0.6, 0.3, 0.1])) if channel == "Rep_F2F" else 0
        interaction_type = "DETAILING_CALL"
    elif channel == "Approved_Email":
        duration_secs = random.randint(5, 60)
        interaction_type = np.random.choice(["DELIVERED", "OPENED", "CLICKED"], p=[0.60, 0.28, 0.12])
    elif channel == "Webinar":
        duration_secs = int(np.random.normal(1800, 400)) # ~30 minutes
        interaction_type = "ATTENDED_SYMPOSIUM"
    else:
        duration_secs = random.randint(1, 15)
        interaction_type = "IMPRESSION"

    engagements.append({
        "engagement_id": f"ENG_{uuid.uuid4().hex[:10].upper()}",
        "hcp_id": h_id,
        "timestamp": event_timestamp.isoformat(),
        "channel": channel,
        "interaction_type": interaction_type,
        "topic": topic,
        "duration_seconds": max(0, duration_secs),
        "sentiment_feedback": feedback_sentiment,
        "samples_dropped_units": samples_dropped
    })

df_engagement = pd.DataFrame(engagements)
df_engagement.to_csv(f"{OUTPUT_DIR}/fct_engagement.csv", index=False)

print("\nFinished generating dataset.")
print(f"Data saved in Apache Parquet format inside ./{OUTPUT_DIR}/")
print(f"Summary:")
print(f" - HCPs: {len(df_hcp)}")
print(f" - Payers: {len(df_payer)}")
print(f" - Patients: {len(df_patient)}")
print(f" - Claims: {len(df_claims)}")
print(f" - Engagement Events: {len(df_engagement)}")

1/5 Generating HCP Dimension...
2/5 Generating Payer & Formulary Dimension...
3/5 Generating Patient Master Cohort...
4/5 Simulating Longitudinal Prescribing & Claims...
5/5 Simulating Omnichannel Field & Digital Engagement...

Finished generating dataset.
Data saved in Apache Parquet format inside ./pharma_data_warehouse/
Summary:
 - HCPs: 250
 - Payers: 8
 - Patients: 2500
 - Claims: 18508
 - Engagement Events: 6000
